# Jupyter-Notebook zum Aktualisieren und alphabetisch Sortieren von ElvUI locale-Dateien

Dieses Notebook unterstützt das Einlesen, Vergleichen und Aktualisieren der `ElvUI_MerathilisUI/ElvUI_MerathilisUI/Locales`-Dateien. Es identifiziert fehlende / veraltete Schlüssel und kann die Einträge alphabetisch sortieren.

In [ ]:
from pathlib import Path
import re
import json
from collections import OrderedDict
import subprocess
import difflib

# Optional imports
try:
    import toml
except ImportError:
    toml = None

try:
    from ruamel.yaml import YAML
except ImportError:
    YAML = None

root_dir = Path('e:/GIT/ElvUI_MerathilisUI/ElvUI_MerathilisUI')
locales_dir = root_dir / 'Locales'

print('Root dir:', root_dir)
print('Locales dir:', locales_dir)

## Repository-Pfade und Dateien finden

Liste alle `.lua`-Dateien im `Locales`-Ordner und filtere auf bekannte Sprachcodes.

In [ ]:
locale_files = sorted([p for p in locales_dir.glob('*.lua') if p.is_file()])

known_codes = {'enUS', 'deDE', 'esMX', 'frFR', 'itIT', 'ptBR', 'ruRU', 'koKR', 'zhCN', 'zhTW'}

print(f'Found {len(locale_files)} locale files:')
for path in locale_files:
    print(' -', path.name)

language_files = {path.stem: path for path in locale_files if path.stem in known_codes}
print('\nDetected language files:')
for code, path in language_files.items():
    print(code, '->', path.name)

## Lua-Locale-Dateien parsen (L[\"key\"] = \"value\")

Implementiere einen Parser, der `L[\"key\"] = value`-Einträge erkennt und Kommentare in separaten Strukturen speichert.

In [ ]:
entry_regex = re.compile(r'''^\t?L\[\"(?P<key>.*?)\"\]\s*=\s*(?P<value>.+)$''')
string_regex = re.compile(r'^\"(?P<text>(?:[^\"\\]|\\.)*)\"$')

def parse_lua_locale(path):
    entries = OrderedDict()
    metadata = []
    raw_lines = path.read_text(encoding='utf-8').splitlines()
    current_block = []

    for line in raw_lines:
        stripped = line.strip()
        if not stripped or stripped.startswith('--'):
            metadata.append(line)
            continue

        match = entry_regex.match(line)
        if not match:
            metadata.append(line)
            continue

        key = match.group('key')
        value = match.group('value').strip()
        entries[key] = value
        current_block.append((key, value))

    return {
        'path': path,
        'raw_lines': raw_lines,
        'entries': entries,
        'metadata': metadata,
        'blocks': current_block,
    }

example_locale = parse_lua_locale(language_files['enUS'])
print('enUS entries:', len(example_locale['entries']))
print('Sample keys:', list(example_locale['entries'].keys())[:10])

## Referenz-Locale laden (z.B. enUS)

Nutze `enUS.lua` als Basis, um fehlende und veraltete Schlüssel zu erkennen.

In [ ]:
reference_code = 'enUS'
reference_locale = parse_lua_locale(language_files[reference_code])
reference_keys = list(reference_locale['entries'].keys())

print('Reference key count:', len(reference_keys))
print('First 20 reference keys:')
for key in reference_keys[:20]:
    print('-', key)

## Fehlende / veraltete Keys erkennen

Vergleiche jede Sprache mit der Referenz und erzeuge einen Bericht.

In [ ]:
def analyze_locale(path, reference_keys):
    locale = parse_lua_locale(path)
    keys = set(locale['entries'].keys())
    missing = [k for k in reference_keys if k not in keys]
    extra = [k for k in keys if k not in reference_keys]
    return {
        'path': path,
        'keys': keys,
        'missing': missing,
        'extra': extra,
        'entry_count': len(keys),
    }

reports = {}
for code, path in language_files.items():
    reports[code] = analyze_locale(path, reference_keys)

for code, report in reports.items():
    print(f"{code}: entries={report['entry_count']}, missing={len(report['missing'])}, extra={len(report['extra'])}")
    if report['missing']:
        print('  Missing sample:', report['missing'][:5])
    if report['extra']:
        print('  Extra sample:', report['extra'][:5])
    print()

# Jupyter-Notebook zum Aktualisieren und alphabetisch Sortieren von ElvUI locale-Dateien

Dieses Notebook unterstützt das Einlesen, Vergleichen und Aktualisieren der `ElvUI_MerathilisUI/ElvUI_MerathilisUI/Locales`-Dateien. Es identifiziert fehlende / veraltete Schlüssel und kann die Einträge alphabetisch sortieren.